## Load useful libraries

In [153]:
import os
import shutil
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
from biological_graph_database.config import ftp_root_ncbi
from biological_graph_database.tools import download_file
from biological_graph_database.tools import compute_file_md5, read_md5_file

## User settings

In [3]:
remote_directory = '/pub/taxonomy'
email_address = os.environ['EMAIL_ADDRESS']
directory_download_to = os.environ['BIOLOGICAL_GRAPH_DATABASE_HOME'] + '/data/taxonomy'

## Create the data directory for the taxonomy download

In [4]:
shutil.rmtree(directory_download_to, ignore_errors = True)
Path(directory_download_to).mkdir()

## Download the taxonomy data

In [5]:
file_to_process = download_file('http://' + ftp_root_ncbi + '/' + remote_directory + '/taxdump.tar.gz', directory_download_to)
file_to_process_md5 = download_file('http://' + ftp_root_ncbi + '/' + remote_directory + '/taxdump.tar.gz.md5', directory_download_to)

## Check whether download succeeded

In [6]:
assert compute_file_md5(file_to_process) == read_md5_file(file_to_process_md5)

## Decompress the downloaded file

In [7]:
directory_working = os.getcwd()
os.chdir(directory_download_to)
os.system('gunzip taxdump.tar.gz')
os.system('tar -xf taxdump.tar')
os.system('gzip taxdump.tar')
os.chdir(directory_working)

In [80]:
import re

with open(directory_download_to + '/readme.txt') as f:
    list_lines = [x.strip() for x in f.readlines()]

new_list_lines = []
for line in list_lines:
    new_line = line
    
    #new_line = line.split('\t')[0].split('--')[0].strip()
    #new_line = line.split('\t')[0].strip()
    
    new_line = re.sub(' +', ' ', new_line)
    new_list_lines.append(new_line)
list_lines = new_list_lines



dict_results = {}
current_dmp_filename = None
for line in list_lines:
    if line.find('.dmp') >= 0 and line.find('--') == -1:

        current_dmp_filename = line.split('.dmp')[0].strip() + '.dmp'
        
        if current_dmp_filename.find('(') >= 0:
            thing = current_dmp_filename.split('(')
            for item in thing:
                if '.dmp' in item:
                    current_dmp_filename = item.strip()

        current_dmp_filename = current_dmp_filename.replace(')', '')
        current_dmp_filename = current_dmp_filename.strip()
        
        dict_results[current_dmp_filename] = []
    else:
        if line.strip() != '':
            to_add = line

            if to_add.find('--') >= 0 or to_add.strip().find('comments') == 0:
                to_add = to_add.split('\t')[0]

                to_add = to_add.split('--')[0]
            
                to_add = re.sub(' +', ' ', to_add)
            
                to_add = to_add.replace('(0 or 1)', '')
                to_add = to_add.replace('(1 or 0)', '')

                to_add = to_add.strip()
                to_add = to_add.replace(' ', '_')


                
                to_add = to_add.strip()
                if current_dmp_filename != None and to_add != '':
                    dict_results[current_dmp_filename].append(to_add)


del(dict_results['*.dmp'])

pp.pprint(dict_results)

{'citations.dmp': ['cit_id',
                   'cit_key',
                   'pubmed_id',
                   'medline_id',
                   'url',
                   'text',
                   'taxid_list'],
 'delnodes.dmp': ['tax_id'],
 'division.dmp': ['division_id', 'division_cde', 'division_name', 'comments'],
 'gencode.dmp': ['genetic_code_id', 'abbreviation', 'name', 'cde', 'starts'],
 'images.dmp': ['image_id',
                'image_key',
                'url',
                'license',
                'attribution',
                'source',
                'properties',
                'taxid_list'],
 'merged.dmp': ['old_tax_id', 'new_tax_id'],
 'names.dmp': ['tax_id', 'name_txt', 'unique_name', 'name_class'],
 'nodes.dmp': ['tax_id',
               'parent_tax_id',
               'rank',
               'embl_code',
               'division_id',
               'inherited_div_flag',
               'genetic_code_id',
               'inherited_GC_flag',
               'mitoc

In [99]:
def load_file_to_df(filename, columns):
    df = pd.read_csv(filename, sep = '\t', header = None)
    df = df.drop(columns = [x for x in range(0, len(df.columns)) if x % 2 == 1])
    df.columns = columns
    return df

In [100]:
df_nodes = load_file_to_df(directory_download_to + '/nodes.dmp', dict_results['nodes.dmp'])
df_nodes.head(3)

,tax_id,parent_tax_id,rank,embl_code,division_id,inherited_div_flag,genetic_code_id,inherited_GC_flag,mitochondrial_genetic_code_id,inherited_MGC_flag,GenBank_hidden_flag,hidden_subtree_root_flag,comments
0,1,1,no rank,NaN,8,0,1,0,0,0,0,0,NaN
1,2,131567,domain,NaN,0,0,11,0,0,0,0,0,NaN
2,6,335928,genus,NaN,0,1,11,1,0,1,0,0,code compliant


In [101]:
assert len(df_nodes.index) == len(df_nodes['tax_id'].unique())

In [108]:
set_tax_ids = set(list(df_nodes['tax_id'].unique()))
set_parent_tax_ids = set(list(df_nodes['parent_tax_id'].unique()))

In [102]:
df_delnodes = load_file_to_df(directory_download_to + '/delnodes.dmp', dict_results['delnodes.dmp'])
df_delnodes.head(3)

,tax_id
0,3712593
1,3712592
2,3712542


In [109]:
set_deleted_tax_ids = set(list(df_delnodes['tax_id'].unique()))

In [107]:
df_merged = load_file_to_df(directory_download_to + '/merged.dmp', dict_results['merged.dmp'])
df_merged.head(3)

,old_tax_id,new_tax_id
0,12,74109
1,30,29
2,36,184914


In [110]:
set_old_tax_ids = set(list(df_merged['old_tax_id'].unique()))
set_new_tax_ids = set(list(df_merged['new_tax_id'].unique()))

In [116]:
df_names = load_file_to_df(directory_download_to + '/names.dmp', dict_results['names.dmp'])
df_names.head(5)

,tax_id,name_txt,unique_name,name_class
0,1,all,NaN,synonym
1,1,root,NaN,scientific name
2,2,Bacteria,Bacteria <bacteria>,scientific name
3,2,bacteria,bacteria <blast name>,blast name
4,2,bacteria,bacteria <genbank common name>,genbank common name


In [117]:
set_names_tax_ids = set(list(df_names['tax_id'].unique()))

In [130]:
df_images = load_file_to_df(directory_download_to + '/images.dmp', dict_results['images.dmp'])
df_images.head(3)

,image_id,image_key,url,license,attribution,source,properties,taxid_list
0,64364,image:Candidatus Aciduliprofundum boonei,http://www.ncbi.nlm.nih.gov/Taxonomy/taxi/imag...,Public Domain (https://creativecommons.org/pub...,NaN,Wikimedia Commons,NaN,379547
1,64365,image:Cyanophora paradoxa,http://www.ncbi.nlm.nih.gov/Taxonomy/taxi/imag...,CC BY-SA 3.0 (https://creativecommons.org/lice...,Wolfgang Bettighofer,Wikimedia Commons,NaN,2762
2,64366,image:Emiliania huxleyi,http://www.ncbi.nlm.nih.gov/Taxonomy/taxi/imag...,CC BY-SA 4.0 (https://creativecommons.org/lice...,Dr. Jeremy Young,Wikimedia Commons,NaN,2903


In [181]:
import re

def process_taxid_list(df, keyname = 'taxid_list'):
    df_temp = df[[keyname]].copy().dropna()
    list_taxid_list = [str(x).strip() for x in list(df_temp[keyname].values)]
    list_taxid_list = [re.sub(' +', ' ', x) for x in list_taxid_list]
    list_taxid_list = [x.split(' ') for x in list_taxid_list]
    list_taxid_list = [x for x in list_taxid_list if len(x) > 0]
    
    new_list = []
    for item in list_taxid_list:
        new_list.append([np.int64(x) for x in item])
    
    return new_list

In [185]:
list_images_taxid_list = process_taxid_list(df_images, 'taxid_list')
list_images_tax_ids = []
for list_of_tax_ids in list_images_taxid_list:
    list_images_tax_ids.extend(list_of_tax_ids)
set_images_tax_ids = set(list_images_tax_ids)

#sorted(list(set_images_tax_ids))[0:5]

[[np.int64(379547)],
 [np.int64(2762)],
 [np.int64(2903)],
 [np.int64(91943)],
 [np.int64(29911)]]

['379547', '2762', '2903', '91943', '29911']

In [126]:
df_citations = load_file_to_df(directory_download_to + '/citations.dmp', dict_results['citations.dmp'])
df_citations.head(5)

ParserError: Error tokenizing data. C error: Expected 14 fields in line 30200, saw 15


In [112]:
set_parent_tax_ids.issubset(set_tax_ids)

True

In [111]:
set_deleted_tax_ids.issubset(set_tax_ids)

False

In [113]:
set_old_tax_ids.issubset(set_tax_ids)

False

In [114]:
set_new_tax_ids.issubset(set_tax_ids)

True

In [118]:
set_tax_ids.issubset(set_names_tax_ids)

False

In [119]:
set_names_tax_ids.issubset(set_tax_ids)

True

In [120]:
set_deleted_tax_ids.issubset(set_names_tax_ids)

False

In [121]:
set_old_tax_ids.issubset(set_names_tax_ids)

False

In [122]:
set_new_tax_ids.issubset(set_names_tax_ids)

False

In [123]:
set_all_tax_ids = set_tax_ids.union(set_names_tax_ids).union(set_deleted_tax_ids).union(set_old_tax_ids).union(set_new_tax_ids)

In [155]:
sorted(list(set_all_tax_ids))[0:10]

[np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10)]